# Healthcare Utilization Pipeline — Demo

Validates the silver entity extraction (patients, encounters, conditions) and gold utilization models across MIMIC and Synthea sources.

In [1]:
import duckdb
import pandas as pd

DB_PATH = "../data/warehouse/mimic_fhir.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)

## 1. Silver Layer — Row Counts by Source

In [2]:
for table in ['patients', 'encounters', 'conditions']:
    df = con.execute(f"SELECT source, COUNT(*) as rows FROM silver.{table} GROUP BY source ORDER BY source").fetchdf()
    print(f"--- silver.{table} ---")
    display(df)
    print()

--- silver.patients ---


,source,rows
0,mimic-iv,299712
1,synthea,107



--- silver.encounters ---


,source,rows
0,mimic-iv,425087
1,synthea,5545



--- silver.conditions ---


,source,rows
0,mimic-iv,899050
1,synthea,3443


## 2. Encounter Classes by Source

Key: MIMIC-IV is all ED (`EMER`). Synthea has a mix — we filter to `EMER` for the index cohort.

In [3]:
con.execute("""
    SELECT source, encounter_class, COUNT(*) as cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY source), 1) as pct
    FROM silver.encounters
    GROUP BY source, encounter_class
    ORDER BY source, cnt DESC
""").fetchdf()

,source,encounter_class,cnt,pct
0,mimic-iv,EMER,425087,100.0
1,synthea,AMB,5187,93.5
2,synthea,EMER,182,3.3
3,synthea,IMP,127,2.3
4,synthea,HH,39,0.7
5,synthea,VR,10,0.2


## 3. Patient Demographics Comparison

In [4]:
con.execute("""
    SELECT source, gender, COUNT(*) as cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY source), 1) as pct
    FROM silver.patients
    GROUP BY source, gender
    ORDER BY source, gender
""").fetchdf()

,source,gender,cnt,pct
0,mimic-iv,female,158553,52.9
1,mimic-iv,male,141159,47.1
2,synthea,female,54,50.5
3,synthea,male,53,49.5


In [5]:
con.execute("""
    SELECT source, race, COUNT(*) as cnt
    FROM silver.patients
    GROUP BY source, race
    ORDER BY source, cnt DESC
    LIMIT 15
""").fetchdf()

,source,race,cnt
0,mimic-iv,White,185877
1,mimic-iv,Black or African American,37478
2,mimic-iv,None,36255
3,mimic-iv,unknown,24231
4,mimic-iv,Asian,13617
5,mimic-iv,asked but unknown,1023
6,mimic-iv,American Indian or Alaska Native,585
7,mimic-iv,Native Hawaiian or Other Pacific Islander,339
8,mimic-iv,other,307
9,synthea,White,93


## 4. Null Rate Check

Key quality metric — high null rates indicate extraction issues.

In [6]:
def null_rates(table, columns):
    cases = ", ".join(
        f"ROUND(100.0 * SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS {c}"
        for c in columns
    )
    return con.execute(f"SELECT source, COUNT(*) as total, {cases} FROM silver.{table} GROUP BY source").fetchdf()

print("--- silver.patients null rates (%) ---")
display(null_rates('patients', ['gender', 'birth_date', 'race', 'ethnicity', 'deceased_datetime']))
print()
print("--- silver.encounters null rates (%) ---")
display(null_rates('encounters', ['patient_id', 'encounter_class', 'period_start', 'period_end', 'status', 'type_code']))
print()
print("--- silver.conditions null rates (%) ---")
display(null_rates('conditions', ['patient_id', 'encounter_id', 'clinical_status', 'category', 'code', 'code_system', 'onset_datetime']))

--- silver.patients null rates (%) ---


,source,total,gender,birth_date,race,ethnicity,deceased_datetime
0,synthea,107,0.0,0.0,0.0,0.0,89.7
1,mimic-iv,299712,0.0,0.0,12.1,20.5,90.3



--- silver.encounters null rates (%) ---


,source,total,patient_id,encounter_class,period_start,period_end,status,type_code
0,synthea,5545,0.0,0.0,0.0,0.0,0.0,0.0
1,mimic-iv,425087,0.0,0.0,0.0,0.0,0.0,0.0



--- silver.conditions null rates (%) ---


,source,total,patient_id,encounter_id,clinical_status,category,code,code_system,onset_datetime
0,synthea,3443,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mimic-iv,899050,0.0,0.0,100.0,0.0,0.0,0.0,100.0


## 5. ED Index Cohort Preview

The gold layer defines the index cohort as all finished EMER encounters joined with patient demographics. Running the query directly here (gold SQLMesh models not yet applied).

In [7]:
ed_cohort = con.execute("""
    SELECT
        e.resource_id AS encounter_id,
        e.source,
        e.patient_id,
        e.period_start,
        e.period_end,
        EXTRACT(EPOCH FROM (e.period_end - e.period_start)) / 3600.0 AS los_hours,
        p.gender,
        p.birth_date,
        DATE_DIFF('year', p.birth_date, CAST(e.period_start AS DATE)) AS age_at_visit,
        p.race,
        p.ethnicity
    FROM silver.encounters e
    JOIN silver.patients p
      ON e.patient_id = p.resource_id AND e.source = p.source
    WHERE e.encounter_class = 'EMER' AND e.status = 'finished'
""").fetchdf()

print(f"ED index cohort: {len(ed_cohort):,} encounters")
print(f"  MIMIC:   {(ed_cohort.source == 'mimic-iv').sum():,}")
print(f"  Synthea: {(ed_cohort.source == 'synthea').sum():,}")
print()
ed_cohort.groupby('source')[['age_at_visit', 'los_hours']].describe().round(1)

ED index cohort: 425,269 encounters


  MIMIC:   425,087


  Synthea: 182



age_at_visit                                            los_hours  \
                count  mean   std   min   25%   50%   75%    max     count   
source                                                                       
mimic-iv     425087.0  52.9  20.6  18.0  35.0  53.0  69.0  103.0  425087.0   
synthea         182.0  35.9  20.2   0.0  20.0  33.5  52.0   90.0     182.0   

                                                  
          mean   std   min  25%  50%  75%    max  
source                                            
mimic-iv   7.2   6.6 -22.7  3.5  5.5  8.3  493.1  
synthea   10.9  51.6   1.0  1.0  1.0  1.0  435.5

## 6. Utilization History Preview

For each ED encounter, count prior encounters in 6/12/24-month lookback windows. This is the core utilization feature.

In [8]:
utilization = con.execute("""
    WITH index_events AS (
        SELECT e.resource_id AS encounter_id, e.source, e.patient_id, e.period_start
        FROM silver.encounters e
        JOIN silver.patients p ON e.patient_id = p.resource_id AND e.source = p.source
        WHERE e.encounter_class = 'EMER' AND e.status = 'finished'
    ),
    prior AS (
        SELECT
            idx.encounter_id AS index_encounter_id,
            idx.source,
            e.encounter_class,
            DATE_DIFF('month', e.period_start, idx.period_start) AS months_before
        FROM index_events idx
        JOIN silver.encounters e
          ON e.patient_id = idx.patient_id
         AND e.source = idx.source
         AND e.period_start < idx.period_start
         AND e.period_start >= idx.period_start - INTERVAL '24' MONTH
    )
    SELECT
        index_encounter_id, source,
        COUNT(*) AS encounters_24m,
        COUNT(*) FILTER (WHERE months_before <= 12) AS encounters_12m,
        COUNT(*) FILTER (WHERE months_before <= 6) AS encounters_6m,
        COUNT(*) FILTER (WHERE encounter_class = 'EMER' AND months_before <= 12) AS ed_visits_12m,
        COUNT(*) FILTER (WHERE encounter_class = 'IMP' AND months_before <= 12) AS inpatient_12m,
        COUNT(*) FILTER (WHERE encounter_class = 'AMB' AND months_before <= 12) AS ambulatory_12m
    FROM prior
    GROUP BY index_encounter_id, source
""").fetchdf()

print(f"Encounters with prior utilization: {len(utilization):,}")
print()
print("--- Mean prior encounters by source ---")
utilization.groupby('source')[['encounters_6m', 'encounters_12m', 'encounters_24m',
                                'ed_visits_12m', 'inpatient_12m', 'ambulatory_12m']].mean().round(2)

Encounters with prior utilization: 194,895

--- Mean prior encounters by source ---


,encounters_6m,encounters_12m,encounters_24m,ed_visits_12m,inpatient_12m,ambulatory_12m
source,,,,,,
mimic-iv,2.02,3.15,4.67,3.15,0.00,0.00
synthea,4.29,6.80,10.62,0.24,0.06,6.47


## 7. Condition Burden Preview

In [9]:
burden = con.execute("""
    WITH index_events AS (
        SELECT e.resource_id AS encounter_id, e.source, e.patient_id, e.period_start
        FROM silver.encounters e
        JOIN silver.patients p ON e.patient_id = p.resource_id AND e.source = p.source
        WHERE e.encounter_class = 'EMER' AND e.status = 'finished'
    )
    SELECT
        idx.encounter_id, idx.source,
        COUNT(DISTINCT c.resource_id) AS total_conditions,
        COUNT(DISTINCT c.resource_id) FILTER (WHERE c.category = 'problem-list-item') AS chronic_conditions,
        COUNT(DISTINCT c.resource_id) FILTER (WHERE c.category = 'encounter-diagnosis') AS encounter_diagnoses,
        COUNT(DISTINCT c.code) AS distinct_codes
    FROM index_events idx
    LEFT JOIN silver.conditions c
      ON c.patient_id = idx.patient_id AND c.source = idx.source
     AND (c.clinical_status = 'active'
          OR c.encounter_id = idx.encounter_id
          OR (c.onset_datetime <= idx.period_start
              AND (c.abatement_datetime IS NULL OR c.abatement_datetime > idx.period_start)))
    GROUP BY idx.encounter_id, idx.source
""").fetchdf()

print(f"Encounters with condition data: {len(burden):,}")
print()
print("--- Mean condition burden by source ---")
burden.groupby('source')[['total_conditions', 'chronic_conditions', 'encounter_diagnoses', 'distinct_codes']].mean().round(2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Encounters with condition data: 425,269

--- Mean condition burden by source ---


,total_conditions,chronic_conditions,encounter_diagnoses,distinct_codes
source,,,,
mimic-iv,2.11,0.0,2.11,2.11
synthea,13.46,0.0,13.46,12.93


## 8. Top Conditions by Source

What diagnoses appear most frequently in each dataset?

In [10]:
for src in ['mimic-iv', 'synthea']:
    print(f"--- Top 10 conditions ({src}) ---")
    display(con.execute(f"""
        SELECT code, code_system, code_display, COUNT(*) as cnt
        FROM silver.conditions
        WHERE source = '{src}'
        GROUP BY code, code_system, code_display
        ORDER BY cnt DESC
        LIMIT 10
    """).fetchdf())
    print()

--- Top 10 conditions (mimic-iv) ---


,code,code_system,code_display,cnt
0,4019,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Unspecified essential hypertension,26816
1,I10,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Essential (primary) hypertension,21264
2,R079,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,"Chest pain, unspecified",13016
3,78650,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,"Chest pain, unspecified",12398
4,25000,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Diabetes mellitus without mention of complicat...,12026
5,R109,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Unspecified abdominal pain,10504
6,E119,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Type 2 diabetes mellitus without complications,8801
7,78909,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,"Abdominal pain, other specified site",8704
8,W1830XA,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,"Fall on same level, unspecified, initial encou...",7817
9,E8889,http://mimic.mit.edu/fhir/mimic/CodeSystem/mim...,Unspecified fall,7081



--- Top 10 conditions (synthea) ---


,code,code_system,code_display,cnt
0,314529007,http://snomed.info/sct,Medication review due (situation),646
1,66383009,http://snomed.info/sct,Gingivitis (disorder),286
2,160903007,http://snomed.info/sct,Full-time employment (finding),244
3,73595000,http://snomed.info/sct,Stress (finding),243
4,160904001,http://snomed.info/sct,Part-time employment (finding),143
5,422650009,http://snomed.info/sct,Social isolation (finding),102
6,444814009,http://snomed.info/sct,Viral sinusitis (disorder),99
7,423315002,http://snomed.info/sct,Limited social contact (finding),89
8,741062008,http://snomed.info/sct,Not in labor force (finding),82
9,18718003,http://snomed.info/sct,Gingival disease (disorder),75


In [11]:
con.close()